In [19]:
# Librerías estándar para manipulación de datos
import pandas as pd
import numpy as np
from pathlib import Path

# Métricas y modelo
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
data_path = Path('../../data/train_features.csv')
# keep_default_na=False desactiva la interpretación automática de nulos
# na_values=[] lista vacía — ningún valor adicional se interpreta como nulo
df = pd.read_csv(data_path, keep_default_na=False, na_values=[''])

print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Nulos totales: {df.isnull().sum().sum()}')
df.columns

Filas: 1460
Columnas: 47
Nulos totales: 0


Index(['OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', 'FullBath',
       'MasVnrArea', 'Fireplaces', 'BsmtFinSF1', 'LotFrontage', 'WoodDeckSF',
       '2ndFlrSF', 'OpenPorchSF', 'HalfBath', 'property_age',
       'year_since_remod', 'ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC',
       'BsmtExposure', 'BsmtFinType1', 'GarageFinish', 'PavedDrive',
       'LotShape', 'Foundation_CBlock', 'Foundation_PConc', 'Foundation_Slab',
       'Foundation_Stone', 'Foundation_Wood', 'GarageType_Attchd',
       'GarageType_Basment', 'GarageType_BuiltIn', 'GarageType_CarPort',
       'GarageType_Detchd', 'GarageType_None', 'MSZoning_FV', 'MSZoning_RH',
       'MSZoning_RL', 'MSZoning_RM', 'SaleCondition_AdjLand',
       'SaleCondition_Alloca', 'SaleCondition_Family', 'SaleCondition_Normal',
       'SaleCondition_Partial', 'CentralAir', 'Neighborhood', 'SalePrice_log'],
      dtype='object')

### Modelo 1: XGBoost - Parámetros Base

Se entrenó un modelo XGBoost con parámetros iniciales como punto de partida
para establecer una referencia antes de la optimización.

**Parámetros utilizados:**
- n_estimators: 500
- max_depth: 4
- learning_rate: 0.05
- random_state: 42

In [22]:
X = df.drop('SalePrice_log', axis = 1)
y = df['SalePrice_log']

print(f'Shape X: {X.shape}')
print(f'Shape Y: {y.shape}')

x_train, x_test, y_train, y_test = train_test_split(
    X,y, test_size = 0.2, random_state = 42
)

print(f"Training Data Set: {x_train.shape[0]} filas")
print(f"Test Data Set: {x_test.shape[0]} filas")

# model_xgb = XGBRegressor(
#     n_estimators=500,
#     learning_rate=0.05,
#     max_depth=4,
#     random_state=42
# )

model_xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model_xgb.fit(x_train, y_train)
y_pred_log = model_xgb.predict(x_test)


# Metricas Sale Price Log
mae_log = mean_absolute_error(y_test,y_pred_log)
mse_log = mean_squared_error(y_test,y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test,y_pred_log))
r2_log = r2_score(y_test, y_pred_log)

print(f"MAE (log): {mae_log}")
print(f"MSE (log): {mse_log}")
print(f"RMSE (log): {rmse_log}")
print(f"R² (log): {r2_log}")

# Reconvesion a escala original
y_pred_usd = np.exp(y_pred_log)
y_test_usd = np.exp(y_test)

# Metricas Sale Price USD
mae_usd = mean_absolute_error(y_test_usd,y_pred_usd)
mse_usd = mean_squared_error(y_test_usd,y_pred_usd)
rmse_usd = np.sqrt(mean_squared_error(y_test_usd,y_pred_usd))
r2_usd = r2_score(y_test_usd, y_pred_usd)

print(f"MAE (USD): {mae_usd}")
print(f"MSE (USD): {mse_usd}")
print(f"RMSE (USD): {rmse_usd}")
print(f"R² (USD): {r2_usd}")



Shape X: (1460, 46)
Shape Y: (1460,)
Training Data Set: 1168 filas
Test Data Set: 292 filas
MAE (log): 0.09283994386788576
MSE (log): 0.02005586985067424
RMSE (log): 0.14161874823156093
R² (log): 0.8925272400468207
MAE (USD): 16032.81485445206
MSE (USD): 683999852.5911196
RMSE (USD): 26153.390843084184
R² (USD): 0.910825193855483


**Resultados:**
| Métrica | Valor |
|---------|-------|
| RMSE (log) | 0.1416 |
| RMSE (USD) | $26,153 |
| R² (USD) | 0.9108 |
| MAE (USD) | $16,033 |

**Conclusión:**
El XGBoost optimizado superó levemente al baseline en USD ($26,153 vs $26,361)
y mejoró el R². Sin embargo, en escala logarítmica (métrica oficial de Kaggle)
el baseline lineal sigue siendo superior (0.1317 vs 0.1416).
El max_depth óptimo de 3 indica que árboles poco profundos generalizan mejor
con este dataset, reduciendo el riesgo de overfitting.

### Modelo 2: XGBoost - Optimizado con GridSearchCV

Se aplicó GridSearchCV para encontrar la combinación óptima de hiperparámetros.
El proceso evaluó 27 combinaciones con cv=5, totalizando 135 entrenamientos.

**Grid de búsqueda:**
| Hiperparámetro | Valores explorados |
|----------------|-------------------|
| n_estimators | 100, 300, 500 |
| max_depth | 3, 4, 5 |
| learning_rate | 0.01, 0.05, 0.1 |

**Mejores parámetros encontrados:**
- n_estimators: 500
- max_depth: 3
- learning_rate: 0.05

In [23]:
params = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5]
}

grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_grid=params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=2,
    verbose=1
)

grid_search.fit(x_train, y_train)
# Mejores parametros encontrados
print(f'Mejores parametros: {grid_search.best_params_}')
print(f'Mejor RMSE (log): {np.sqrt(-grid_search.best_score_):.4f}')

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Mejores parametros: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 500}
Mejor RMSE (log): 0.1353


### Modelo 3: Ridge Regression con RidgeCV

Se exploró Ridge Regression como alternativa al XGBoost. Ridge aplica
regularización L2 penalizando coeficientes grandes para mejorar la
generalización del modelo.

**Configuración:**
- Alphas explorados: [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
- cv: 5

**Mejor alpha encontrado:** 10.0


In [21]:
ridge_cv =RidgeCV(
    alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
    cv=5
)

ridge_cv.fit(x_train, y_train)
print(f'Mejor alpha: {ridge_cv.alpha_}')

y_pred_log_ridge = ridge_cv.predict(x_test)
y_pred_ridge_usd = np.exp(y_pred_log_ridge)

# Metricas Sale Price Log
mae_ridge_log = mean_absolute_error(y_test,y_pred_log_ridge)
mse_ridge_log = mean_squared_error(y_test,y_pred_log_ridge)
rmse_ridge_log = np.sqrt(mean_squared_error(y_test,y_pred_log_ridge))
r2_ridge_log = r2_score(y_test, y_pred_log_ridge)
print(f"MAE Ridge (log): {mae_ridge_log}")
print(f"MSE Ridge (log): {mse_ridge_log}")
print(f"RMSE Ridge (log): {rmse_ridge_log}")
print(f"R² Ridge (log): {r2_ridge_log}")

# Metricas Sale Price USD
mae_ridge_usd = mean_absolute_error(y_test_usd,y_pred_ridge_usd)
mse_ridge_usd = mean_squared_error(y_test_usd,y_pred_ridge_usd)
rmse_ridge_usd = np.sqrt(mean_squared_error(y_test_usd,y_pred_ridge_usd))
r2_ridge_usd = r2_score(y_test_usd, y_pred_ridge_usd)
print(f"MAE Ridge (USD): {mae_ridge_usd}")
print(f"MSE Ridge (USD): {mse_ridge_usd}")
print(f"RMSE Ridge (USD): {rmse_ridge_usd}")
print(f"R² Ridge (USD): {r2_ridge_usd}")

Mejor alpha: 10.0
MAE Ridge (log): 0.09870765041164406
MSE Ridge (log): 0.019467346496422365
RMSE Ridge (log): 0.13952543315260615
R² Ridge (log): 0.8956809416638176
MAE Ridge (USD): 17165.97051408224
MSE Ridge (USD): 698207537.488034
RMSE Ridge (USD): 26423.61704021677
R² Ridge (USD): 0.9089729017217271


**Resultados:**
| Métrica | Valor |
|---------|-------|
| RMSE (log) | 0.1395 |
| RMSE (USD) | $26,424 |
| R² (USD) | 0.9090 |
| MAE (USD) | $17,166 |

**Conclusión:**
El alpha óptimo de 10.0 indica que el dataset requiere una penalización
considerable, confirmando la presencia de coeficientes grandes que
afectaban al modelo lineal simple. Ridge mejora al XGBoost en RMSE log
pero no supera al baseline lineal.

### Comparación Final de Modelos

| Modelo | RMSE (log) | RMSE (USD) | R² (USD) |
|--------|-----------|------------|---------|
| Baseline Lineal | **0.1317** | $26,361 | 0.909 |
| XGBoost Optimizado | 0.1416 | **$26,153** | **0.911** |
| Ridge (α=10) | 0.1395 | $26,424 | 0.909 |
| Benchmark Kaggle | 0.1358 | - | - |

**Modelo seleccionado para submit:** Baseline Lineal
El modelo de regresión lineal obtiene el mejor RMSE en escala logarítmica,
que es la métrica oficial de Kaggle. Su simplicidad y estabilidad lo hacen
la opción más confiable para la predicción final.